# YuvaIntern Task 4 — Data Science Tool Mastery Project
## Iris Dataset Exploration and Classification

**Tools:** Python, Pandas, NumPy, Matplotlib, Seaborn, scikit-learn  
**Dataset:** Iris dataset bundled with scikit-learn

Objective: demonstrate data inspection, data manipulation, numerical summaries, visualization, and a basic classification workflow. The dataset is loaded directly from scikit-learn; no separate CSV download is required.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

sns.set_theme()
iris = load_iris(as_frame=True)
df = iris.frame.copy()
df = df.rename(columns={
    "sepal length (cm)": "sepal_length_cm",
    "sepal width (cm)": "sepal_width_cm",
    "petal length (cm)": "petal_length_cm",
    "petal width (cm)": "petal_width_cm"
})
df["species"] = df["target"].map(dict(enumerate(iris.target_names)))
display(df.head())
print("Shape:", df.shape)


## 1. Data inspection and quality checks

In [ ]:
print("Data types:")
display(df.dtypes.to_frame("dtype"))
print("Descriptive statistics:")
display(df.describe().T)
quality = pd.DataFrame({
    "missing_count": df.isna().sum(),
    "missing_percent": (df.isna().mean() * 100).round(2),
    "unique_values": df.nunique()
})
display(quality)
print("Duplicate rows:", df.duplicated().sum())


## 2. NumPy numerical operations

In [ ]:
features = ["sepal_length_cm", "sepal_width_cm", "petal_length_cm", "petal_width_cm"]
arr = df[features].to_numpy()
numpy_summary = pd.DataFrame({
    "mean": np.mean(arr, axis=0),
    "std": np.std(arr, axis=0),
    "min": np.min(arr, axis=0),
    "max": np.max(arr, axis=0)
}, index=features)
display(numpy_summary)
print("NumPy array shape:", arr.shape)


## 3. Pandas filtering, grouping, and feature creation

In [ ]:
df["petal_to_sepal_length_ratio"] = df["petal_length_cm"] / df["sepal_length_cm"]
summary = df.groupby("species").agg(
    sample_count=("species", "size"),
    avg_petal_length=("petal_length_cm", "mean"),
    avg_petal_width=("petal_width_cm", "mean"),
    avg_sepal_length=("sepal_length_cm", "mean"),
    avg_ratio=("petal_to_sepal_length_ratio", "mean")
).round(3)
display(summary)
filtered = df.loc[df["petal_length_cm"] > 5, ["species", "petal_length_cm", "petal_width_cm"]]
print("Rows with petal length > 5 cm:", len(filtered))
display(filtered.head())


## 4. Visualizations

In [ ]:
avg_petal = df.groupby("species")["petal_length_cm"].mean().sort_values()
plt.figure(figsize=(8, 5))
plt.bar(avg_petal.index, avg_petal.values)
plt.title("Average Petal Length by Species")
plt.xlabel("Species"); plt.ylabel("Average petal length (cm)")
plt.tight_layout(); plt.show()


In [ ]:
plt.figure(figsize=(8, 5))
plt.hist(df["sepal_length_cm"], bins=15, edgecolor="black")
plt.title("Distribution of Sepal Length")
plt.xlabel("Sepal length (cm)"); plt.ylabel("Frequency")
plt.tight_layout(); plt.show()


In [ ]:
plt.figure(figsize=(8, 5))
sns.scatterplot(data=df, x="petal_length_cm", y="petal_width_cm", hue="species")
plt.title("Petal Length vs Petal Width by Species")
plt.tight_layout(); plt.show()


In [ ]:
corr = df[features].corr()
plt.figure(figsize=(7, 5))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm")
plt.title("Feature Correlation Matrix")
plt.tight_layout(); plt.show()


## 5. Basic machine-learning workflow

In [ ]:
X = df[features]
y = df["species"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)
model = Pipeline([
    ("scaler", StandardScaler()),
    ("knn", KNeighborsClassifier(n_neighbors=5))
])
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
print(f"Test accuracy: {accuracy_score(y_test, y_pred):.3f}")
print(classification_report(y_test, y_pred))


In [ ]:
labels = list(iris.target_names)
cm = confusion_matrix(y_test, y_pred, labels=labels)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=labels, yticklabels=labels)
plt.title("KNN Confusion Matrix")
plt.xlabel("Predicted label"); plt.ylabel("True label")
plt.tight_layout(); plt.show()


## 6. Findings and conclusion

- Species-level group summaries compare flower measurements.
- The scatter plot helps inspect whether petal measurements separate species.
- The correlation matrix summarizes linear relationships between numeric features.
- The confusion matrix shows correct predictions and class confusions.
- Test accuracy applies to this specific train/test split and is not a guarantee of future performance.

This notebook demonstrates the practical roles of Pandas, NumPy, Matplotlib, Seaborn and scikit-learn across data loading, inspection, aggregation, visualization, scaling, classification and evaluation.

## Dataset reference
The Iris dataset is distributed with scikit-learn: https://scikit-learn.org/stable/modules/generated/sklearn.datasets.load_iris.html  
Original reference: Fisher, R. A. (1936). *The use of multiple measurements in taxonomic problems*. Annals of Eugenics, 7(2), 179–188.